# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook shows how to load and explore the FAIR^2 dataset using the `mlcroissant` library, referencing entities by their `@id`.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Display summary
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List record sets and their fields by @id
record_sets = dataset.record_sets
print("Record sets available:")
for rs in record_sets:
    print(f"- RecordSet @id: {rs['@id']}, name: {rs.get('name', '[no name]')}")

# For each record set, list its fields
for rs in record_sets:
    print(f"\nRecordSet {rs['@id']} fields:")
    for field in rs.get('field', []):
        print(f"  Field @id: {field['@id']}, name: {field.get('name', '[no name]')}, dataType: {field.get('dataType', '[unknown]')}")

## 3. Data Extraction
Load data from record sets using their `@id` into pandas DataFrames for analysis.

In [ ]:
# Gather record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    # Load records for this record set
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nColumns for RecordSet {record_set_id}: {df.columns.tolist()}")
    print(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply filtering, normalization, grouping, and basic statistics.

In this example, we select suitable numeric and categorical fields based on available metadata.

In [ ]:
# Choose a record set and fields for EDA (update @id and field as needed based on above overview):
# For illustration, we'll select the first record set and try numeric/categorical fields.
target_record_set_id = record_set_ids[0] if record_set_ids else None

df = dataframes.get(target_record_set_id)
if df is not None:
    # Try to find a numeric field (float/integer) automatically
    numeric_field_id = None
    group_field_id = None
    for rs in record_sets:
        if rs['@id'] == target_record_set_id:
            for field in rs.get('field', []):
                dt = str(field.get('dataType', '')).lower()
                if ('float' in dt or 'integer' in dt) and field['@id'] in df.columns:
                    numeric_field_id = field['@id']
                    break
            for field in rs.get('field', []):
                dt = str(field.get('dataType', '')).lower()
                if 'text' in dt or 'string' in dt or 'category' in dt:
                    if field['@id'] in df.columns:
                        group_field_id = field['@id']
                        break

    # If not found, fallback to first columns
    if numeric_field_id is None and len(df.columns) > 0:
        numeric_field_id = df.columns[0]
    if group_field_id is None and len(df.columns) > 1:
        group_field_id = df.columns[1]

    threshold = 10
    if numeric_field_id in df.columns:
        # Remove missing values
        df_num = df[pd.to_numeric(df[numeric_field_id], errors='coerce').notnull()].copy()
        df_num[numeric_field_id] = pd.to_numeric(df_num[numeric_field_id], errors='coerce')
        filtered_df = df_num[df_num[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by a categorical/text field if available
        if group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
            print(grouped_df.head())
    else:
        print("No numeric field found for EDA.")
else:
    print("No records available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields—referencing fields by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot results if numeric and group fields exist
if df is not None and numeric_field_id in df.columns:
    plt.figure(figsize=(8,6))
    sns.histplot(df[numeric_field_id], bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
This notebook demonstrates how to load, reference, and explore the FAIR^2 dataset using `mlcroissant`, referencing entities by their `@id`.

- You can extend this workflow by referencing more record sets and fields via their IDs.
- Filtering, normalization, grouping, and visualization steps prepare the data for model development or policy analysis.

### Key Observations:
- The dataset captures ordered logistic regression output and survey responses for rangeland management in Northern Kenya.
- Data fields, columns, and entities are referenced by their unique `@id`.
- The workflow is adaptable to any FAIR-adherent dataset described by Croissant schema.
